# 30: What Is Digital Life?

This final notebook is an executable evidence synthesis. It does not introduce a new experiment and it does not conclude that digital life has been proved. The verdict boundary is `PROVISIONAL_SPECIFICATION`.

In [1]:
from pathlib import Path
import json
import math
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_PROFILE = os.environ.get("NOTEBOOK_PROFILE", "quick")
RUN_CANONICAL = os.environ.get("RUN_CANONICAL", "0") == "1"


def find_repo_root(start=Path.cwd()):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "content" / "books" / "digital-life").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
FIG_DIR = NOTEBOOK_DIR / "generated-figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_json(relative_path):
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def require_path(relative_path):
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def summarize_result(label, result, status=None, source="canonical research artifact"):
    row = {"label": label, "source": source}
    if status is not None:
        row["status"] = status
    for key in ["n", "mean", "ci95_low", "ci95_high", "achieved_mde80_one_sided"]:
        if key in result:
            row[key] = result[key]
    return row

print("profile", NOTEBOOK_PROFILE, "run_canonical", RUN_CANONICAL)
print("repo", REPO_ROOT)

CHAPTER = 30
MANUSCRIPT = require_path("content/books/digital-life/30-what-is-digital-life/index.md")
print("manuscript", MANUSCRIPT.relative_to(REPO_ROOT))

profile quick run_canonical False
repo C:\Projects\working-book
manuscript content\books\digital-life\30-what-is-digital-life\index.md


## Evidence Matrix

This matrix maps provisional substrate-first statements to prior chapters and committed artifacts where practical. The check below verifies that synthesis claims have provenance and that rejected biological interpretations do not become prerequisites.

In [2]:
evidence_rows = [
    {"property": "ongoing state transition", "status": "SUPPORTED", "chapters": "11,14,20", "artifacts": "notebooks/generated-figures/ch11-crystal-growth-curve.png; research/digital-life/ch20-material-loss-v1/stage-02-loss-sweep.json", "survived": "continued construction and transformation", "not_justified": "life proved"},
    {"property": "turnover / loss creates opportunity", "status": "SUPPORTED BUT NARROW", "chapters": "20", "artifacts": "research/digital-life/ch20-material-loss-v1/stage-02-loss-sweep.json", "survived": "loss can create new frontier opportunity", "not_justified": "repair or metabolism"},
    {"property": "local causal structure", "status": "SUPPORTED", "chapters": "22,23,24", "artifacts": "research/digital-life/ch23-persistent-transient-causal-gain-v4/stage-04-verdict.json; research/digital-life/ch24-causal-accounting-v5/stage-06-verdict.json", "survived": "local intervention has structured consequences", "not_justified": "privileged body boundary"},
    {"property": "finite-computation non-local coupling", "status": "SUPPORTED", "chapters": "25", "artifacts": "research/digital-life/ch25-finite-budget-redistribution-v1/stage-09-verdict.json", "survived": "finite evaluation slots couple distant opportunities", "not_justified": "unbounded dynamics behave the same"},
    {"property": "causal routing changes without resolved amplification", "status": "BOUNDED_NEAR_ZERO + MECHANISM SURVIVES", "chapters": "26", "artifacts": "research/digital-life/ch26-dynamically-matched-rate-causal-amplification-v2/stage-04-primary-test.json; research/digital-life/ch26-v2-mechanism-audit/ch26-v2-mechanism-audit-report.json", "survived": "routing changes across pathways", "not_justified": "aggregate amplification"},
    {"property": "hidden state modulates later response", "status": "DIRECTION_SUPPORTED / MAGNITUDE_UNRESOLVED", "chapters": "27", "artifacts": "research/digital-life/ch27-decaying-material-history-causal-response-v2/stage-04-primary.json", "survived": "same visible geometry can respond differently", "not_justified": "memory"},
    {"property": "raw causal containment", "status": "RAW SUPPORTED; EXCESS BOUNDED BELOW SEI", "chapters": "28", "artifacts": "research/digital-life/ch28-causal-modularity-v1/stage-03-primary.json; research/digital-life/ch28-causal-modularity-v2/stage-04-primary.json", "survived": "raw containment is real", "not_justified": "privileged individuality"},
    {"property": "failure ledger discipline", "status": "FAILURE_LEDGER_CONSISTENT", "chapters": "29", "artifacts": "research/digital-life/ch29-how-to-fail-correctly-v1/stage-05-verdict.json", "survived": "invalid/unresolved/bounded/supported distinctions preserved", "not_justified": "making every experiment successful"},
]
evidence = pd.DataFrame(evidence_rows)
evidence

,property,status,chapters,artifacts,survived,not_justified
0,ongoing state transition,SUPPORTED,"11,14,20",notebooks/generated-figures/ch11-crystal-growt...,continued construction and transformation,life proved
1,turnover / loss creates opportunity,SUPPORTED BUT NARROW,20,research/digital-life/ch20-material-loss-v1/st...,loss can create new frontier opportunity,repair or metabolism
2,local causal structure,SUPPORTED,"22,23,24",research/digital-life/ch23-persistent-transien...,local intervention has structured consequences,privileged body boundary
3,finite-computation non-local coupling,SUPPORTED,25,research/digital-life/ch25-finite-budget-redis...,finite evaluation slots couple distant opportu...,unbounded dynamics behave the same
4,causal routing changes without resolved amplif...,BOUNDED_NEAR_ZERO + MECHANISM SURVIVES,26,research/digital-life/ch26-dynamically-matched...,routing changes across pathways,aggregate amplification
5,hidden state modulates later response,DIRECTION_SUPPORTED / MAGNITUDE_UNRESOLVED,27,research/digital-life/ch27-decaying-material-h...,same visible geometry can respond differently,memory
6,raw causal containment,RAW SUPPORTED; EXCESS BOUNDED BELOW SEI,28,research/digital-life/ch28-causal-modularity-v...,raw containment is real,privileged individuality
7,failure ledger discipline,FAILURE_LEDGER_CONSISTENT,29,research/digital-life/ch29-how-to-fail-correct...,invalid/unresolved/bounded/supported distincti...,making every experiment successful


In [3]:
def artifact_exists_list(value):
    paths = [p.strip() for p in value.split(";")]
    return all((REPO_ROOT / p).exists() for p in paths)

evidence["artifact_paths_exist"] = evidence["artifacts"].apply(artifact_exists_list)
assert evidence["property"].notna().all()
assert evidence["chapters"].notna().all()
assert evidence["artifact_paths_exist"].all()
for forbidden in ["memory", "repair", "metabolism", "privileged individuality", "life proved"]:
    assert forbidden not in "; ".join(evidence["property"].str.lower())
evidence[["property", "status", "chapters", "artifact_paths_exist"]]

,property,status,chapters,artifact_paths_exist
0,ongoing state transition,SUPPORTED,"11,14,20",True
1,turnover / loss creates opportunity,SUPPORTED BUT NARROW,20,True
2,local causal structure,SUPPORTED,"22,23,24",True
3,finite-computation non-local coupling,SUPPORTED,25,True
4,causal routing changes without resolved amplif...,BOUNDED_NEAR_ZERO + MECHANISM SURVIVES,26,True
5,hidden state modulates later response,DIRECTION_SUPPORTED / MAGNITUDE_UNRESOLVED,27,True
6,raw causal containment,RAW SUPPORTED; EXCESS BOUNDED BELOW SEI,28,True
7,failure ledger discipline,FAILURE_LEDGER_CONSISTENT,29,True


In [4]:
status_order = evidence["status"].value_counts().reset_index()
status_order.columns = ["status", "count"]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(status_order["status"], status_order["count"], color="#3978c5")
ax.set_title("DESCRIPTIVE SYNTHESIS: evidence-status inventory")
ax.set_xlabel("matrix entries")
path = FIG_DIR / "ch30-evidence-status-inventory.png"
fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)
path.relative_to(REPO_ROOT)

WindowsPath('notebooks/generated-figures/ch30-evidence-status-inventory.png')

## Provisional Specification

A digital living process may be a computational process that:

1. persists through ongoing state transition rather than fixed material identity;
2. maintains structured causal interaction over time;
3. supports continued construction, turnover or transformation;
4. carries consequences of prior state into future state;
5. allows internal state to redirect later trajectories;
6. operates under finite computational opportunity;
7. need not begin with a privileged biological-style individual boundary.

Compact form: a digital living process is a computational process that persists through ongoing state transition while preserving enough causal organization for prior interactions to constrain future possibilities.

In [5]:
spec = [
    "persists through ongoing state transition",
    "maintains structured causal interaction over time",
    "supports continued construction, turnover or transformation",
    "carries consequences of prior state into future state",
    "allows internal state to redirect later trajectories",
    "operates under finite computational opportunity",
    "need not begin with a privileged biological-style individual boundary",
]
coverage = pd.DataFrame([
    {"spec_statement": spec[0], "mapped_properties": "ongoing state transition; raw causal containment"},
    {"spec_statement": spec[1], "mapped_properties": "local causal structure; raw causal containment"},
    {"spec_statement": spec[2], "mapped_properties": "ongoing state transition; turnover / loss creates opportunity"},
    {"spec_statement": spec[3], "mapped_properties": "hidden state modulates later response; causal routing changes without resolved amplification"},
    {"spec_statement": spec[4], "mapped_properties": "hidden state modulates later response"},
    {"spec_statement": spec[5], "mapped_properties": "finite-computation non-local coupling"},
    {"spec_statement": spec[6], "mapped_properties": "raw causal containment"},
])
assert coverage["mapped_properties"].str.len().gt(0).all()
coverage

,spec_statement,mapped_properties
0,persists through ongoing state transition,ongoing state transition; raw causal containment
1,maintains structured causal interaction over time,local causal structure; raw causal containment
2,"supports continued construction, turnover or t...",ongoing state transition; turnover / loss crea...
3,carries consequences of prior state into futur...,hidden state modulates later response; causal ...
4,allows internal state to redirect later trajec...,hidden state modulates later response
5,operates under finite computational opportunity,finite-computation non-local coupling
6,need not begin with a privileged biological-st...,raw causal containment


## Final Verdict

`PROVISIONAL_SPECIFICATION`, not `SUPPORTED_DIGITAL_LIFE`.

The synthesis preserves what survived and keeps absent biological interpretations absent: repair, readable memory, sender-specific signalling, privileged individuality, reproduction as necessity, and metabolism as a prerequisite.